packages import

In [151]:
import requests
import os
import yaml
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
import pandas as pd

# apollo sracper

In [ ]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"
response = requests.get(url, auth=credentials)

In [ ]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
auth = HTTPBasicAuth(username, password)

In [ ]:
response = requests.get(url, auth=auth)
print(response.status_code)

200


In [ ]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [ ]:
group_tag = page_dom.select_one("div.grupa")
group = group_tag.get_text().strip() if group_tag else "Not found"
print(group)

ZICSS1-1212


In [ ]:
classes_tag = page_dom.select_one("table")

with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag))

classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [ ]:
classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"]), "Typ"] = classes['Typ'].str.capitalize()
print(classes.columns.tolist())

['Termin', 'DzieÅ\x84, godzina', 'Przedmiot', 'Typ', 'Nauczyciel', 'Sala']


In [ ]:
classes[['Day','Start time', 'hyphen', 'End time', "Duration"]] = classes.iloc[:, 1].str.split(' ', expand=True)

In [ ]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [ ]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [ ]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [ ]:
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8-sig", index=False)